In [0]:
from pyspark.sql import functions as F, DataFrame
from datetime import datetime

# ── Configuração ──────────────────────────────────────────────────────────────
SOURCE_TABLE = "homologacao.salutar_saude.raw_empresas"
TARGET_TABLE = "homologacao.salutar_saude.silver_empresas"

DATE_COLS    = ["data_inicio_relacionamento"]
INITCAP_COLS = ["setor", "porte"]          # initcap + trim: cobre variantes de case
TRIM_COLS    = ["empresa_nome"]      # trim apenas
UF_COLS      = ["uf"]               # normaliza para sigla de 2 letras (ex.: "São Paulo" → "SP")
MERGE_KEY    = "empresa_id"         # chave primária para o MERGE incremental

# Mapa nome completo → sigla UF (chaves em UPPER para lookup case-insensitive)
UF_MAP = {
    "ACRE": "AC", "ALAGOAS": "AL", "AMAPÁ": "AP", "AMAZONAS": "AM",
    "BAHIA": "BA", "CEARÁ": "CE", "DISTRITO FEDERAL": "DF",
    "ESPÍRITO SANTO": "ES", "GOIÁS": "GO", "MARANHÃO": "MA",
    "MATO GROSSO": "MT", "MATO GROSSO DO SUL": "MS", "MINAS GERAIS": "MG",
    "PARÁ": "PA", "PARAÍBA": "PB", "PARANÁ": "PR", "PERNAMBUCO": "PE",
    "PIAUÍ": "PI", "RIO DE JANEIRO": "RJ", "RIO GRANDE DO NORTE": "RN",
    "RIO GRANDE DO SUL": "RS", "RONDÔNIA": "RO", "RORAIMA": "RR",
    "SANTA CATARINA": "SC", "SÃO PAULO": "SP", "SERGIPE": "SE",
    "TOCANTINS": "TO",
}

run_ts = datetime.now()

print(f"Pipeline  : silver_empresas")
print(f"Iniciado  : {run_ts:%Y-%m-%d %H:%M:%S}")
print(f"Origem    : {SOURCE_TABLE}")
print(f"Destino   : {TARGET_TABLE}")

Pipeline  : silver_empresas
Iniciado  : 2026-07-04 20:44:31
Origem    : homologacao.salutar_saude.raw_empresas
Destino   : homologacao.salutar_saude.silver_empresas


In [0]:
df_raw = spark.table(SOURCE_TABLE)
n_raw = df_raw.count()

print(f"[OK] {n_raw:,} registros lidos de {SOURCE_TABLE}")

[OK] 41 registros lidos de homologacao.salutar_saude.raw_empresas


In [0]:
def parse_date(col_name: str) -> F.Column:
    """Normaliza datas para YYYY-MM-DD.
    Suporta os formatos: YYYY-MM-DD, DD-MM-YYYY, DD/MM/YYYY.
    Usa try_to_date para retornar NULL em vez de lançar exceção.
    """
    return F.date_format(
        F.coalesce(
            F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd-MM-yyyy')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd/MM/yyyy')"),
        ),
        "yyyy-MM-dd",
    ).alias(col_name)


def parse_initcap(col_name: str) -> F.Column:
    """Normaliza campos categóricos: remove espaços extras e padroniza capitalização.
    Genérico e reutilizável para qualquer campo categórico (porte, setor, status...).
    Não usar em siglas/códigos como UF ("SP" viraria "Sp").
    Exemplos: 'GRANDE' → 'Grande' | 'média ' → 'Média' | 'pequena' → 'Pequena'
    """
    return F.initcap(F.trim(F.col(col_name))).alias(col_name)


def parse_trim(col_name: str) -> F.Column:
    """Remove espaços extras de campos de texto livres (nome)."""
    return F.trim(F.col(col_name)).alias(col_name)


def parse_uf(col_name: str) -> F.Column:
    """Normaliza UF para sigla de 2 letras maiúsculas via UF_MAP.
    - Nomes completos: 'São Paulo' → 'SP', 'Minas Gerais' → 'MG'
    - Siglas já corretas: 'SP' → 'SP', 'sp' → 'SP'
    - Lookup é case-insensitive (F.upper antes de consultar o mapa).
    - Valores não reconhecidos: retornam em UPPER sem falha (detectados pelo DQ assert).
    """
    upper_val = F.upper(F.trim(F.col(col_name)))
    map_pairs = []
    for k, v in UF_MAP.items():
        map_pairs.extend([F.lit(k), F.lit(v)])
    uf_map_expr = F.create_map(*map_pairs)
    return F.coalesce(uf_map_expr[upper_val], upper_val).alias(col_name)


def transform(df: DataFrame, cols: list) -> DataFrame:
    """Aplica todas as transformações de padronização da camada silver.
    Remove a coluna _rescued_data (artefato da camada RAW) e
    adiciona _updated_at com o timestamp de execução.

    Args:
        df  : DataFrame de origem (camada RAW).
        cols: lista de colunas pré-computada fora da função (evita RPC repetido
              ao acessar df.columns dentro de um loop/função no Spark Connect).
    """
    return df.select(
        *[
            parse_date(c)    if c in DATE_COLS    else
            parse_initcap(c) if c in INITCAP_COLS else
            parse_uf(c)      if c in UF_COLS      else
            parse_trim(c)    if c in TRIM_COLS    else
            F.col(c)
            for c in cols
        ],
        F.lit(run_ts).cast("timestamp").alias("_updated_at"),  # em select, não withColumn
    )


print("[OK] Funções de transformação definidas.")

[OK] Funções de transformação definidas.


In [0]:
# Computa schema uma única vez fora da função (evita RPC repetido no Spark Connect)
silver_cols = [c for c in df_raw.columns if c != "_rescued_data"]
df_silver = transform(df_raw, silver_cols)

# ── Validação de qualidade ────────────────────────────────────────────────────
DQ_COLS = DATE_COLS + INITCAP_COLS

null_counts = (
    df_silver
    .select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in DQ_COLS])
    .collect()[0]
    .asDict()
)
unexpected_porte = df_silver.filter(
    F.col("porte").isNotNull()
    & ~F.col("porte").isin("Pequena", "Média", "Grande")
).count()

# Verifica se todos os valores de UF são siglas válidas após normalização
valid_ufs = set(UF_MAP.values())
unexpected_uf = df_silver.filter(
    F.col("uf").isNotNull()
    & ~F.col("uf").isin(*valid_ufs)
).count()

n_silver = df_silver.count()

# Duplicatas de chave na fonte (serão removidas no passo de escrita)
dup_keys_df  = df_silver.groupBy(MERGE_KEY).count().filter(F.col("count") > 1)
n_dup_keys   = dup_keys_df.count()
dup_ids_list = [str(r[MERGE_KEY]) for r in dup_keys_df.collect()] if n_dup_keys > 0 else []

print("── Data Quality ───────────────────────────────────────────────────────────")
print(f"  Total de registros           : {n_silver:,}")
print(f"  Correspondência com RAW      : {'[OK]' if n_silver == n_raw else '[WARN] divergência!'}")
for col_name, nulls in null_counts.items():
    tag = "[WARN]" if nulls > 0 else "[OK]  "
    print(f"  {tag} {col_name}: {nulls:,} nulos")
tag = "[WARN]" if unexpected_porte else "[OK]  "
print(f"  {tag} porte valores inesperados  : {unexpected_porte:,}")
tag = "[WARN]" if unexpected_uf else "[OK]  "
print(f"  {tag} uf valores inválidos (não é sigla UF)      : {unexpected_uf:,}")
tag = "[WARN]" if n_dup_keys > 0 else "[OK]  "
detail = f" → ids: {dup_ids_list}" if dup_ids_list else ""
print(f"  {tag} {MERGE_KEY} duplicados na fonte           : {n_dup_keys:,}{detail}")
print("─" * 65)

assert n_silver == n_raw, f"Contagem divergente: RAW={n_raw}, silver={n_silver}"
assert unexpected_porte == 0, f"{unexpected_porte} valores inesperados em 'porte'"
assert unexpected_uf == 0, f"{unexpected_uf} valores de 'uf' não são siglas UF válidas"

── Data Quality ───────────────────────────────────────────────────────────
  Total de registros           : 41
  Correspondência com RAW      : [OK]
  [OK]   data_inicio_relacionamento: 0 nulos
  [OK]   setor: 0 nulos
  [OK]   porte: 0 nulos
  [OK]   porte valores inesperados  : 0
  [OK]   uf valores inválidos (não é sigla UF)      : 0
  [WARN] empresa_id duplicados na fonte           : 1 → ids: ['8']
─────────────────────────────────────────────────────────────────


In [0]:
from delta.tables import DeltaTable

# ── Escrita incremental via MERGE ─────────────────────────────────────────────
# Remove duplicatas: mantém apenas o registro com empresa_id único (primeira ocorrência)
df_to_merge = df_silver.dropDuplicates([MERGE_KEY])

if spark.catalog.tableExists(TARGET_TABLE):
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_target.alias("target")
        .merge(
            df_to_merge.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}"
        )
        .whenMatchedUpdateAll()       # atualiza empresa existente se houver mudança
        .whenNotMatchedInsertAll()    # insere nova empresa
        .execute()
    )
    print(f"[OK] MERGE concluído      : {TARGET_TABLE}")
else:
    # Primeira execução: cria a tabela (carga inicial)
    df_to_merge.write.format("delta").saveAsTable(TARGET_TABLE)
    print(f"[OK] Carga inicial        : {TARGET_TABLE}")

n_written = spark.table(TARGET_TABLE).count()
print(f"[OK] Registros na tabela  : {n_written:,}")
print(f"[OK] Fim do pipeline      : {datetime.now():%Y-%m-%d %H:%M:%S}")

[OK] Carga inicial        : homologacao.salutar_saude.silver_empresas
[OK] Registros na tabela  : 40
[OK] Fim do pipeline      : 2026-07-04 20:44:39


In [0]:
%sql
SELECT * FROM homologacao.salutar_saude.silver_empresas
ORDER BY empresa_id

empresa_id,empresa_nome,setor,porte,uf,data_inicio_relacionamento,_updated_at
1,Empresa A01 Ltda,Saúde,Pequena,SP,2023-05-18,2026-07-04T20:44:31.987Z
2,Empresa B02 Ltda,Indústria,Grande,MG,2022-06-21,2026-07-04T20:44:31.987Z
3,Empresa C03 Ltda,Tecnologia,Pequena,BA,2024-02-23,2026-07-04T20:44:31.987Z
4,Empresa D04 Ltda,Indústria,Média,MG,2023-01-27,2026-07-04T20:44:31.987Z
5,Empresa E05 Ltda,Tecnologia,Média,PR,2023-01-17,2026-07-04T20:44:31.987Z
6,Empresa F06 Ltda,Varejo,Média,RS,2024-09-27,2026-07-04T20:44:31.987Z
7,Empresa G07 Ltda,Indústria,Pequena,BA,2024-06-25,2026-07-04T20:44:31.987Z
8,Empresa H08 Ltda,Indústria,Grande,MG,2021-02-20,2026-07-04T20:44:31.987Z
9,Empresa I09 Ltda,Serviços Financeiros,Média,BA,2024-10-03,2026-07-04T20:44:31.987Z
10,Empresa J10 Ltda,Agronegócio,Pequena,RJ,2021-03-21,2026-07-04T20:44:31.987Z
